In [4]:
import numpy as np
import pandas as pd

In [5]:
proxy = pd.read_csv("../proxy/realized_cov_h21.csv", parse_dates=["Date"]).set_index("Date").sort_index()
ewma  = pd.read_csv("../models/ewma_cov_lambda094.csv", parse_dates=["Date"]).set_index("Date").sort_index()
dcc   = pd.read_csv("../results/dcc_cov_forecast_h21.csv", parse_dates=["Date"]).set_index("Date").sort_index()

In [6]:
print("Proxy shape:", proxy.shape)
print("EWMA shape:", ewma.shape)
print("DCC shape:", dcc.shape)

print("Proxy vs EWMA same columns:", list(proxy.columns) == list(ewma.columns))
print("Proxy vs DCC same columns:", list(proxy.columns) == list(dcc.columns))

common_ewma = proxy.index.intersection(ewma.index)
common_dcc  = proxy.index.intersection(dcc.index)

print("EWMA common dates:", len(common_ewma), common_ewma.min().date(), "->", common_ewma.max().date())
print("DCC common dates:", len(common_dcc), common_dcc.min().date(), "->", common_dcc.max().date())

Proxy shape: (2184, 64)
EWMA shape: (1255, 64)
DCC shape: (1255, 64)
Proxy vs EWMA same columns: True
Proxy vs DCC same columns: True
EWMA common dates: 1235 2021-01-04 -> 2025-12-02
DCC common dates: 1235 2021-01-04 -> 2025-12-02


In [7]:
def row_to_matrix(row, n_assets=8):
    mat = row.to_numpy(dtype=float).reshape(n_assets, n_assets)
    mat = 0.5 * (mat + mat.T)
    return mat

def make_spd(mat, eps=1e-10):
    mat = 0.5 * (mat + mat.T)
    mat = mat + eps * np.eye(mat.shape[0])
    return mat

def qlike_loss(proxy_mat, forecast_mat, eps=1e-10):
    S = make_spd(proxy_mat, eps=eps)
    H = make_spd(forecast_mat, eps=eps)

    n = S.shape[0]

    H_inv = np.linalg.inv(H)
    trace_term = np.trace(S @ H_inv)

    sign_s, logdet_s = np.linalg.slogdet(S)
    sign_h, logdet_h = np.linalg.slogdet(H)

    if sign_s <= 0 or sign_h <= 0:
        raise ValueError("Non-positive determinant encountered in QLIKE calculation.")

    return float(trace_term - (logdet_s - logdet_h) - n)

def frobenius_loss(proxy_mat, forecast_mat):
    diff = forecast_mat - proxy_mat
    return float(np.sum(diff ** 2))

def evaluate_against_proxy(forecast_df, proxy_df, model_name, n_assets=8):
    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates].copy()
    proxy_df = proxy_df.loc[common_dates].copy()

    if list(forecast_df.columns) != list(proxy_df.columns):
        raise ValueError(f"Column mismatch between {model_name} and proxy.")

    records = []

    for dt in common_dates:
        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets=n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets=n_assets)

        records.append({
            "Date": dt,
            "QLIKE": qlike_loss(S_mat, H_mat),
            "FROBENIUS": frobenius_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_qlike": losses["QLIKE"].mean(),
        "avg_frobenius": losses["FROBENIUS"].mean()
    })

    return losses, summary

In [9]:
ewma_losses, ewma_summary = evaluate_against_proxy(ewma, proxy, model_name="EWMA")
dcc_losses, dcc_summary = evaluate_against_proxy(dcc, proxy, model_name="DCC")

summary_table = pd.DataFrame([ewma_summary, dcc_summary]).reset_index(drop=True)
summary_table

,model,n_obs,start_date,end_date,avg_qlike,avg_frobenius
0,EWMA,1235,2021-01-04,2025-12-02,5.235591,0.000002
1,DCC,1235,2021-01-04,2025-12-02,4.488435,0.000017


In [11]:
loss_compare = pd.DataFrame({
    "EWMA_QLIKE": ewma_losses["QLIKE"],
    "DCC_QLIKE": dcc_losses["QLIKE"],
    "EWMA_FROB": ewma_losses["FROBENIUS"],
    "DCC_FROB": dcc_losses["FROBENIUS"],
})

loss_compare.describe()

print("DCC better on QLIKE share:",
      (dcc_losses["QLIKE"] < ewma_losses["QLIKE"]).mean())

print("DCC better on Frobenius share:",
      (dcc_losses["FROBENIUS"] < ewma_losses["FROBENIUS"]).mean())

DCC better on QLIKE share: 0.725506072874494
DCC better on Frobenius share: 0.3595141700404858
